# CrossVul-DG — Results Analysis

Loads the per-fold / per-seed `results.json` files produced by
`scripts/run_lopo.py`, aggregates macro-F1 per method, and reproduces the
figures used in the paper (cluster-count ablation and per-method F1).

> **Note.** Until you run `scripts/run_full_sweep.sh` on real MegaVul data,
> the `runs/` directory will be empty and this notebook falls back to the
> tiny synthetic sanity-check run. The numbers in the paper tables are
> *illustrative placeholders* and must be replaced with the output of this
> notebook on real runs.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

RUNS_DIR = Path("../runs")          # adjust if needed
METHOD_ORDER = ["erm", "da-coral", "da-dann", "irm", "dro", "pd-dro"]
METHOD_LABEL = {
    "erm": "ERM-FT", "da-coral": "DA-CORAL", "da-dann": "DA-DANN",
    "irm": "CrossVul-DG (IRM)", "dro": "CrossVul-DG (DRO)",
    "pd-dro": "CrossVul-DG (PD+DRO)",
}

In [ ]:
def load_runs(runs_dir: Path):
    """method -> held_out -> seed -> test metrics."""
    bucket = defaultdict(lambda: defaultdict(dict))
    for p in runs_dir.rglob("results.json"):
        with open(p) as f:
            r = json.load(f)
        a = r["args"]
        bucket[a["method"]][a["held_out"]][a["seed"]] = r["test"]
    return bucket

bucket = load_runs(RUNS_DIR)
print(f"Found runs for methods: {sorted(bucket)}" if bucket
      else f"No runs found under {RUNS_DIR.resolve()} -- run the sweep first.")

In [ ]:
def aggregate_f1(bucket):
    """Return {method: (mean_f1, std_f1, n_seeds)} averaging over folds then seeds."""
    out = {}
    for method, folds in bucket.items():
        seeds = {s for f in folds.values() for s in f}
        seed_means = []
        for seed in seeds:
            per_fold = [f[seed]["f1"] for f in folds.values() if seed in f]
            if per_fold:
                seed_means.append(np.mean(per_fold))
        if seed_means:
            out[method] = (float(np.mean(seed_means)),
                           float(np.std(seed_means)), len(seed_means))
    return out

agg = aggregate_f1(bucket)
for m in METHOD_ORDER:
    if m in agg:
        mu, sd, n = agg[m]
        print(f"{METHOD_LABEL[m]:<22} F1 = {mu*100:5.1f} +/- {sd*100:.1f}  (seeds={n})")

In [ ]:
# --- Figure: macro-F1 per method (cross-project LOPO) ---
methods = [m for m in METHOD_ORDER if m in agg]
if methods:
    means = [agg[m][0] * 100 for m in methods]
    errs  = [agg[m][1] * 100 for m in methods]
    plt.figure(figsize=(7, 4))
    plt.bar(range(len(methods)), means, yerr=errs, capsize=4)
    plt.xticks(range(len(methods)), [METHOD_LABEL[m] for m in methods],
               rotation=30, ha="right")
    plt.ylabel("Cross-project macro-F1 (%)")
    plt.title("LOPO macro-F1 by method")
    plt.tight_layout()
    plt.savefig("../figures/f1_by_method.pdf", bbox_inches="tight")
    plt.show()
else:
    print("No aggregated results to plot yet.")

In [ ]:
# --- Figure: pseudo-domain cluster-count (C) ablation ---
# Expects runs named with the pd-dro method under different configs/n_clusters,
# e.g. runs/pd-dro_C{C}/... ; adapt the parser to your sweep naming.
def cluster_ablation(runs_dir: Path):
    by_c = defaultdict(list)
    for p in runs_dir.rglob("results.json"):
        with open(p) as f:
            r = json.load(f)
        cfg = r["args"].get("config", {})
        C = cfg.get("pseudo", {}).get("n_clusters")
        if r["args"]["method"] == "pd-dro" and C is not None:
            by_c[C].append(r["test"]["f1"])
    return {c: float(np.mean(v)) for c, v in by_c.items() if v}

abl = cluster_ablation(RUNS_DIR)
if abl:
    cs = sorted(abl)
    plt.figure(figsize=(6, 4))
    plt.plot(cs, [abl[c] * 100 for c in cs], marker="o")
    plt.xlabel("Number of pseudo-domains C")
    plt.ylabel("Macro-F1 (%)")
    plt.title("Pseudo-domain cluster-count ablation")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("../figures/cluster_ablation.pdf", bbox_inches="tight")
    plt.show()
else:
    print("No pd-dro ablation runs found (vary pseudo.n_clusters in the sweep).")

## Reproducing the paper tables

Run `python scripts/aggregate_results.py --runs_dir runs/ --out paper_tables/`
to write `paper_tables/main_table.json`, then paste the aggregated values
into `main.tex` (replacing the illustrative placeholders in Tables II and
III).